# 03 · Task 2 — tuning the step size

*A learning-rate sweep for CRNTiny*

Neural Audio and Speech Processing — Day 2 / Application to Speech Enhancement
Homework assignment, R. Scheibler (2026-07-02) · dataset: Voicebank-DEMAND (16 kHz)

> **Task 2.** Tune the step size `lr`, i.e. find the best value.

The baseline uses `lr=1e-3` with AdamW (`wd=0.02`), 500 linear warmup steps and a
cosine decay to zero over the whole run — so `lr` is the *peak* of the schedule,
not a constant.

**Protocol.** A logarithmic grid, one short run each, everything else fixed
(seed 42, batch 32, `leaky_relu`). The winner is retrained at full length. Because
the schedule is cosine-annealed over `epochs`, each short run is self-consistent:
it completes its own decay rather than being truncated.

**Runtime:** 5 × ~10 min + one full run ≈ 1.7 h on a T4.

In [ ]:
# --- Google Colab bootstrap (does nothing when running locally) --------------
# IMPORTANT: point this at the fork that contains the homework modifications
# (models/crn.py with a selectable activation, train.py with --warmup-steps).
REPO_URL = "https://github.com/Ahmed-AlGhosaini/nanoSE.git"

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os

    if not os.path.exists("nanoSE"):
        !git clone -q $REPO_URL nanoSE
    %cd nanoSE
    !pip install -q -r requirements.txt

    import torch
    if not torch.cuda.is_available():
        print("No GPU! Runtime > Change runtime type > T4 GPU, then re-run this cell.")

In [ ]:
import sys
from pathlib import Path

# Make notebooks/nb_utils.py importable no matter where the kernel was started
for candidate in (Path.cwd(), Path.cwd() / "notebooks", Path.cwd().parent / "notebooks"):
    if (candidate / "nb_utils.py").exists():
        sys.path.insert(0, str(candidate))
        break

import matplotlib.pyplot as plt
import pandas as pd

import nb_utils

ROOT = nb_utils.bootstrap()   # chdir to the repository root + print the device

In [ ]:
SWEEP_EPOCHS = 5
FULL_EPOCHS = 25

LR_GRID = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]   # 1e-3 is the baseline

> **Resuming after a disconnect.** Every finished run is recorded in
> `notebooks/experiment_runs.json`, and the sweep loops skip anything already recorded.
> Re-running the cell after a Colab timeout continues where it stopped instead of
> starting over.

In [ ]:
lr_runs = {}
for lr in LR_GRID:
    key = f"lr_{lr:g}"
    done = nb_utils.recall(key)
    if done is not None:
        lr_runs[lr] = done
        print(f"[skip] lr={lr:<8g} already trained -> {done}")
        continue

    config = nb_utils.write_config(
        f"exp_{key}.py",
        name=key,
        model="CRNTiny()",
        docstring=f"Task 2: peak learning rate {lr:g}.",
        epochs=SWEEP_EPOCHS,
        lr=lr,
    )
    lr_runs[lr] = nb_utils.remember(key, nb_utils.run_training(config))

In [ ]:
labels = [f"lr={lr:g}" for lr in LR_GRID]
runs = [lr_runs[lr] for lr in LR_GRID]

table = nb_utils.summarize(runs, labels=labels, best_epoch=True)
table[["label", "lr", "epoch", "val_si_sdr", "pesq", "estoi", "dnsmos"]].round(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.8))

nb_utils.plot_curves(runs, labels=labels, metric="val_si_sdr", ax=axes[0],
                     title="Validation SI-SDR per epoch")

# Response curve: one series, so no legend -- the title names it
ordered = table.set_index("lr").reindex(LR_GRID)
axes[1].plot(LR_GRID, ordered.val_si_sdr, color=nb_utils.PALETTE[0], linewidth=2,
             marker="o", markersize=7)
axes[1].set_xscale("log")
for lr, value in zip(LR_GRID, ordered.val_si_sdr):
    axes[1].annotate(f"{value:.2f}", (lr, value), xytext=(0, 8), textcoords="offset points",
                     ha="center", color=nb_utils.INK_SOFT, fontsize=9)
nb_utils.style_axes(axes[1], f"Best SI-SDR within {SWEEP_EPOCHS} epochs vs peak learning rate",
                    "Peak learning rate (log scale)", "Validation SI-SDR (dB)")
plt.tight_layout()
plt.show()

In [ ]:
# Training loss curves expose divergence that the validation metrics only hint at.
# (The loss can be negative -- it contains the -SI-SDR term -- so no log axis here.)
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
nb_utils.plot_curves(runs, labels=labels, metric="loss", ax=axes[0],
                     title="Training loss per epoch")
nb_utils.plot_curves(runs, labels=labels, metric="train_si_sdr", ax=axes[1],
                     title="Training SI-SDR per epoch")
plt.tight_layout()
plt.show()

## Full-length run at the best learning rate

In [ ]:
best_lr = float(table.iloc[0].lr)
print(f"best learning rate in the short sweep: {best_lr:g}")

if best_lr == 1e-3:
    print("The baseline learning rate won -- reusing the baseline run.")
    lr_full_run = nb_utils.recall("baseline")
else:
    lr_full_run = nb_utils.recall("lr_best_full")
    if lr_full_run is None:
        config = nb_utils.write_config(
            "exp_lr_best_full.py",
            name=f"lr_{best_lr:g}_full",
            model="CRNTiny()",
            docstring=f"Task 2 winner: peak learning rate {best_lr:g}, full-length run.",
            epochs=FULL_EPOCHS,
            lr=best_lr,
        )
        lr_full_run = nb_utils.remember("lr_best_full", nb_utils.run_training(config))

nb_utils.remember("best_lr_value", best_lr)   # picked up by notebooks 04 and 05
print("full-length run:", lr_full_run)

In [ ]:
baseline_run = nb_utils.recall("baseline")
comparison = [r for r in (baseline_run, lr_full_run) if r is not None]
comparison_labels = ["baseline (lr=1e-3)", f"lr={best_lr:g}"][: len(comparison)]
nb_utils.summarize(comparison, labels=comparison_labels).round(3)

## Discussion

*Fill in with your numbers.* What to look for:

* **The shape of the response curve.** A good `lr` sweep is an inverted U on a log
  axis. If the maximum sits at the edge of the grid, the grid is too narrow — extend
  it before concluding.
* **Too small** (`1e-4`): the run is not diverging, it is simply *unfinished* — the
  loss is still falling when the cosine schedule reaches zero.
* **Too large** (`1e-2`): watch the training-loss plot. Instability shows up there
  epochs before the validation metrics collapse.
* **Careful with short-run conclusions.** A large `lr` looks better early and worse
  late; this is exactly why the winner is confirmed with a full-length run rather
  than promoted straight from the sweep.

**Next:** `04_speed_and_single_epoch.ipynb` (Tasks 3 & 4).